# Repetições individuais — PlantVillage
Selecione **um modelo e uma semente por execução**. Usa os scripts existentes, batch 32,
até 30 épocas, Adam, early stopping patience 5/min_delta 1e-4 e o mesmo CSV de splits.
A seed muda inicialização, augmentations e ordem de treino; não refaz a divisão.
A seed 42 permanece nos diretórios originais. As novas execuções ficam em
`results/repetitions/seed_<semente>/<modelo>/`. Diretórios existentes são bloqueados.
Apenas treino e validação. Nenhuma célula avalia imagens de teste.

No Colab: selecione runtime com GPU L4, monte o Drive abaixo e atualize **src/**
com esta versão do projeto. Use a pasta que contém os resultados L4 originais da seed 42.
O notebook não instala nem atualiza dependências automaticamente: mantenha as versões
PyTorch, torchvision e CUDA registradas na referência. Não execute os notebooks de split.


In [10]:
from pathlib import Path
import sys
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

PROJECT_ROOT = (Path('/content/drive/MyDrive/plant-disease-classification')
                if IN_COLAB else Path.cwd())
if not (PROJECT_ROOT / 'src').is_dir() and (PROJECT_ROOT.parent / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
IMAGE_ROOT = Path('/content/plantvillage_color') if IN_COLAB else PROJECT_ROOT / 'data/plantvillage_color'
ZIP_PATH = PROJECT_ROOT / "data" / "data.zip"

assert ZIP_PATH.is_file(), f"ZIP não encontrado: {ZIP_PATH}"
SRC_DIR = PROJECT_ROOT / 'src'
print('Projeto:', PROJECT_ROOT)
print('Scripts:', SRC_DIR)
required_scripts = ('run_repetition.py', 'compare_repetitions.py', 'compare_models.py',
                    'experiment_utils.py', 'plantvillage_pytorch.py',
                    'train_baseline.py', 'train_resnet50.py',
                    'train_densenet201.py', 'train_efficientnet_b0.py')
missing_scripts = [name for name in required_scripts if not (SRC_DIR / name).is_file()]
if missing_scripts:
    raise FileNotFoundError(
        f"Scripts ausentes em {SRC_DIR}: {', '.join(missing_scripts)}. "
        "Abrir o notebook pelo GitHub não atualiza os arquivos no Drive. "
        "Ajuste PROJECT_ROOT para o projeto correto ou atualize o clone com "
        f'git -C "{PROJECT_ROOT}" pull --ff-only. '
        "Se a pasta não for um clone Git, copie a pasta src atualizada para ela. "
        "Depois reexecute o notebook desde esta célula. Não instale run_repetition com pip."
    )
sys.path.insert(0, str(SRC_DIR))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Projeto: /content/drive/MyDrive/plant-disease-classification
Scripts: /content/drive/MyDrive/plant-disease-classification/src


In [11]:
MODEL = 'densenet201'  # baseline_cnn, resnet50, densenet201, efficientnet_b0
SEED = 123  # 123 ou 456
EXECUTE_TRAINING = True  # altere para True somente quando desejar iniciar


## Imagens locais (opcional)
Se as imagens já estão extraídas, ajuste IMAGE_ROOT e pule a próxima célula.
Caso contrário, ela extrai do ZIP **somente os caminhos de treino e validação**
do CSV existente. Não baixa dados, não gera splits e não abre imagens de teste.


In [12]:
EXTRACT_IMAGES = True  # True somente se precisar extrair do ZIP
if EXTRACT_IMAGES:
    import pandas as pd
    import shutil
    import zipfile
    from plantvillage_pytorch import _raw_color_relative_parts
    metadata = pd.read_csv(PROJECT_ROOT / 'results/plantvillage_metadata_split.csv')
    selected = metadata.loc[metadata['split'].isin(['train', 'validation']), 'zip_path']
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for member in selected:
            parts = _raw_color_relative_parts(member)
            target = IMAGE_ROOT.joinpath(*parts)
            if not target.resolve().is_relative_to(IMAGE_ROOT.resolve()):
                raise ValueError('Caminho inválido')
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open('wb') as destination:
                    shutil.copyfileobj(source, destination)
    print('Extração de treino e validação concluída.')


Extração de treino e validação concluída.


In [13]:
import json
import importlib
import sys

# Recheck the selected root even if the configuration cell was edited later.
SRC_DIR = PROJECT_ROOT / 'src'
if not (SRC_DIR / 'run_repetition.py').is_file():
    raise FileNotFoundError(
        f"Não encontrei {SRC_DIR / 'run_repetition.py'}. "
        "Execute novamente a célula de caminhos e atualize a pasta src do projeto no Drive. "
        "Abrir o notebook no Colab não sincroniza os scripts do GitHub."
    )
sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()
from run_repetition import prepare_repetition, run_repetition
command, manifest = prepare_repetition(PROJECT_ROOT, MODEL, SEED, IMAGE_ROOT)
print(json.dumps(manifest, indent=2, ensure_ascii=False))


{
  "model": "densenet201",
  "seed": 123,
  "reference_seed": 42,
  "metadata_csv_sha256": "844d4a333c55024570d0e5c69b0e5a2e70d3f221f9c41289f6a992a067ff1bcd",
  "reference_metadata_sha256": "6d39582d72ea61d5e115e0df85aac8fda6edd69daaae2de58f85a8dad76f0416",
  "evaluation_split": "validation",
  "test_split_used": false,
  "command": [
    "/usr/bin/python3",
    "/content/drive/MyDrive/plant-disease-classification/src/train_densenet201.py",
    "--metadata-csv",
    "/content/drive/MyDrive/plant-disease-classification/results/plantvillage_metadata_split.csv",
    "--image-root",
    "/content/plantvillage_color",
    "--output-dir",
    "/content/drive/MyDrive/plant-disease-classification/results/repetitions/seed_123/densenet201",
    "--seed",
    "123",
    "--batch-size",
    "32",
    "--epochs",
    "30",
    "--patience",
    "5",
    "--min-delta",
    "0.0001",
    "--num-classes",
    "38",
    "--num-workers",
    "2",
    "--lr",
    "0.0001"
  ],
  "output_dir": "/content/

In [14]:
if EXECUTE_TRAINING:
    output = run_repetition(PROJECT_ROOT, MODEL, SEED, IMAGE_ROOT)
    print('Execução concluída:', output)
else:
    print('Somente preparação. Para treinar, defina EXECUTE_TRAINING = True e execute esta célula.')


Execução concluída: /content/drive/MyDrive/plant-disease-classification/results/repetitions/seed_123/densenet201


## Comparação de todas as execuções disponíveis
Lê apenas artefatos salvos. Não treina e não carrega checkpoints.
A seed 42 é lida do diretório original. As seeds 123 e 456 são lidas dos novos diretórios.
O desvio padrão é **amostral (ddof=1)** entre sementes; com uma execução fica indefinido (NaN).
`n_runs` mostra a quantidade válida: a comparação só fica completa com 3 seeds por modelo.
Resultados ausentes/inválidos não entram nas médias. Avisos de hardware/ambiente devem
ser resolvidos antes de interpretar tempos como uma comparação controlada.


In [15]:
from compare_repetitions import compare_repetitions
comparison = compare_repetitions(PROJECT_ROOT / 'results')
display(comparison.individual)
display(comparison.aggregate)
for warning in comparison.warnings:
    print(warning)


/content/drive/MyDrive/plant-disease-classification/src/compare_repetitions.py:69: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  individual = pd.concat(frames, ignore_index=True)
/content/drive/MyDrive/plant-disease-classification/src/compare_repetitions.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  per_class = pd.concat(classes, ignore_index=True)


,model_key,model_name,status,missing_artifacts,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,...,throughput_images_per_second,gpu,gpu_total_memory_gb,selected_device,pytorch_version,cuda_version,batch_size,num_workers,input_size,seed
0,baseline_cnn,CNN baseline,complete,,0.961821,0.954637,0.958447,0.955367,0.963451,0.961821,...,723.151146,NVIDIA L4,23.659151,cuda,2.11.0+cu130,13.0,32,2,3x224x224,42
1,resnet50,ResNet50,complete,,0.994860,0.992530,0.991833,0.992097,0.994912,0.994860,...,697.143757,NVIDIA L4,23.659151,cuda,2.11.0+cu130,13.0,32,2,3x224x224,42
2,densenet201,DenseNet201,complete,,0.994616,0.991986,0.994109,0.992959,0.994735,0.994616,...,408.461050,NVIDIA L4,23.659151,cuda,2.11.0+cu130,13.0,32,2,3x224x224,42
3,efficientnet_b0,EfficientNet-B0,complete,,0.996207,0.992885,0.994838,0.993820,0.996247,0.996207,...,725.797374,NVIDIA L4,23.659151,cuda,2.11.0+cu130,13.0,32,2,3x224x224,42
4,baseline_cnn,CNN baseline,complete,,0.958027,0.952218,0.949185,0.949471,0.959075,0.958027,...,664.151523,NVIDIA L4,23.659151,cuda,2.11.0+cu130,13.0,32,2,3x224x224,123
5,resnet50,ResNet50,complete,,0.994493,0.990955,0.993589,0.992197,0.994599,0.994493,...,679.801482,NVIDIA L4,23.659151,cuda,2.11.0+cu130,13.0,32,2,3x224x224,123
6,densenet201,DenseNet201,complete,,0.993147,0.991104,0.992152,0.991594,0.993214,0.993147,...,406.296403,NVIDIA L4,23.659151,cuda,2.11.0+cu130,13.0,32,2,3x224x224,123
7,efficientnet_b0,EfficientNet-B0,pending,efficientnet_b0_best.pth; efficientnet_b0_hist...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,None,NaN,None,None,None,<NA>,<NA>,None,123
8,baseline_cnn,CNN baseline,pending,baseline_cnn_best.pth; baseline_cnn_history.cs...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,None,NaN,None,None,None,<NA>,<NA>,None,456
9,resnet50,ResNet50,pending,resnet50_best.pth; resnet50_history.csv; valid...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,None,NaN,None,None,None,<NA>,<NA>,None,456


,model_key,model_name,n_runs,expected_runs,seeds,status,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,...,average_time_per_epoch_seconds_mean,average_time_per_epoch_seconds_std,inference_time_seconds_mean,inference_time_seconds_std,inference_seconds_per_image_mean,inference_seconds_per_image_std,inference_milliseconds_per_image_mean,inference_milliseconds_per_image_std,throughput_images_per_second_mean,throughput_images_per_second_std
0,baseline_cnn,CNN baseline,2,3,42;123,partial,0.959924,0.002682,0.953427,0.001710,...,95.808879,3.758633,11.802482,0.709850,0.001444,0.000087,1.444259,0.086864,693.651335,41.719033
1,resnet50,ResNet50,2,3,42;123,partial,0.994677,0.000260,0.991743,0.001114,...,186.893625,0.575227,11.871636,0.211454,0.001453,0.000026,1.452721,0.025875,688.472620,12.262840
2,densenet201,DenseNet201,2,3,42;123,partial,0.993882,0.001038,0.991545,0.000624,...,329.125161,0.502189,20.060099,0.075371,0.002455,0.000009,2.454736,0.009223,407.378727,1.530636
3,efficientnet_b0,EfficientNet-B0,1,3,42,partial,0.996207,NaN,0.992885,NaN,...,115.053510,NaN,11.259341,NaN,0.001378,NaN,1.377795,NaN,725.797374,NaN


seed 123: EfficientNet-B0: pendente/incompleto; faltam: efficientnet_b0_best.pth, efficientnet_b0_history.csv, validation_metrics.csv, validation_metrics_by_class.csv, validation_confusion_matrix.png, architecture_summary.json, timing_summary.json, experiment_metadata.json.
seed 456: CNN baseline: pendente/incompleto; faltam: baseline_cnn_best.pth, baseline_cnn_history.csv, validation_metrics.csv, validation_metrics_by_class.csv, validation_confusion_matrix.png, architecture_summary.json, timing_summary.json, experiment_metadata.json.
seed 456: ResNet50: pendente/incompleto; faltam: resnet50_best.pth, resnet50_history.csv, validation_metrics.csv, validation_metrics_by_class.csv, validation_confusion_matrix.png, architecture_summary.json, timing_summary.json, experiment_metadata.json.
seed 456: DenseNet201: pendente/incompleto; faltam: densenet201_best.pth, densenet201_history.csv, validation_metrics.csv, validation_metrics_by_class.csv, validation_confusion_matrix.png, architecture_sum